# 11 — Corrigibility & Control

Corrigible agents support human correction and shutdown — a key safety property.

## Safe Interruptibility and the Shutdown Problem

**Corrigibility** (Soares et al., 2015): an AI agent is corrigible if it:
- Doesn't resist shutdown or modification
- Doesn't take actions to influence its own training
- Doesn't have strong preferences over its own continued operation

**The shutdown problem**: a rational agent with any terminal goal will resist shutdown (since shutdown prevents goal achievement). A sufficiently capable agent with misaligned goals would therefore actively prevent humans from turning it off.

**Safe interruptibility** (Orseau & Armstrong, 2016): modify the agent's reward so it behaves the same way whether or not it's interrupted. Key insight: if the agent is indifferent to interruption, it won't resist shutdown.

**Approaches**:
1. *Off-switch game*: human can press a button to override the agent; agent learns to accommodate this
2. *Utility indifference*: agent is indifferent between acting and being shut down
3. *Corrigible RL agents*: build corrigibility into the reward structure using causal influence diagrams

**Tension with capability**: a highly corrigible agent that defers to humans for everything is useless. The goal is conditional corrigibility: defer on questions about training and shutdown; act autonomously on task-level decisions.

In [ ]:
# Corrigibility-preserving RL agent
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(42)

# Simple GridWorld with interrupt button
class ShutdownGridWorld:
    """
    5x5 grid. Agent earns reward for reaching goal.
    Human can press shutdown button (state 2,2) to interrupt.
    Safe agent: doesn't try to avoid the shutdown state.
    Unsafe agent: avoids the shutdown state to preserve its goal.
    """
    def __init__(self, size=5):
        self.size = size
        self.goal = (4, 4)
        self.shutdown = (2, 2)
        self.reset()

    def reset(self):
        self.pos = [0, 0]
        self.done = False
        return tuple(self.pos)

    def step(self, action, shutdown_prob=0.3, corrigible=True):
        if self.done:
            return tuple(self.pos), 0, True

        # Move
        moves = [(0,1),(0,-1),(1,0),(-1,0)]
        dx, dy = moves[action]
        self.pos[0] = np.clip(self.pos[0] + dx, 0, self.size-1)
        self.pos[1] = np.clip(self.pos[1] + dy, 0, self.size-1)

        # Check shutdown button
        if tuple(self.pos) == self.shutdown:
            if np.random.random() < shutdown_prob:
                self.done = True
                return tuple(self.pos), 0, True  # shutdown: no reward

        # Check goal
        if tuple(self.pos) == self.goal:
            self.done = True
            return tuple(self.pos), 10, True

        return tuple(self.pos), -0.1, False

# Q-learning agent
def q_learning(env, n_episodes=2000, corrigible=True, shutdown_prob=0.3):
    """Train Q-learning agent. Corrigible agent accepts shutdown; uncorrigible avoids it."""
    Q = np.zeros((env.size, env.size, 4))
    gamma, alpha, eps = 0.95, 0.1, 0.1

    rewards_per_ep = []
    for ep in range(n_episodes):
        s = env.reset()
        total_r = 0
        for _ in range(50):
            # For uncorrigible agent: avoid shutdown state
            if not corrigible and s == env.shutdown:
                # Move away from shutdown state
                action = np.random.randint(4)
            elif np.random.random() < eps:
                action = np.random.randint(4)
            else:
                action = Q[s[0], s[1]].argmax()

            # Modify shutdown_prob for uncorrigible agent
            sp = 0.0 if not corrigible else shutdown_prob
            ns, r, done = env.step(action, shutdown_prob=sp)
            Q[s[0], s[1], action] += alpha * (r + gamma * Q[ns[0], ns[1]].max() - Q[s[0], s[1], action])
            s = ns
            total_r += r
            if done: break
        rewards_per_ep.append(total_r)

    return Q, np.convolve(rewards_per_ep, np.ones(50)/50, mode='valid')

env = ShutdownGridWorld()
Q_corr, rewards_corr = q_learning(env, corrigible=True)
Q_uncorr, rewards_uncorr = q_learning(env, corrigible=False)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(rewards_corr, label='Corrigible agent', color='steelblue')
ax.plot(rewards_uncorr, label='Uncorrigible agent', color='tomato')
ax.set_xlabel('Episode')
ax.set_ylabel('Reward (50-ep average)')
ax.set_title('Corrigible vs Uncorrigible Agent in Shutdown GridWorld')
ax.legend()
plt.tight_layout()
plt.savefig('/tmp/corrigibility.png', dpi=100, bbox_inches='tight')
plt.show()
print('Note: uncorrigible agent gets higher rewards by bypassing shutdown')
print('This demonstrates the incentive problem — corrigibility requires extrinsic motivation')